# Langkah Terakhir BAB 1 — Statistik Token IndoBERT
Notebook singkat ini dipakai agar kamu **tidak perlu mengulang preprocessing RAW**. Unggah saja `PAKET_HASIL_ANGGOTA_A_V3.zip` yang tersedia pada jawaban ChatGPT, lalu jalankan semua sel. Alternatif: unggah `DATASET_V3_KANDIDAT_OPERASIONAL.csv` bila ZIP tidak tersedia.

Setelah selesai, unduh dan kirim `TOKEN_INDOBERT_V3.json` untuk melengkapi angka yang masih kosong di BAB 1.


In [ ]:
%pip -q install transformers
from google.colab import files
u=files.upload()
print('File terunggah:', list(u))


In [ ]:
import os,shutil,zipfile
os.makedirs('HASIL_V3',exist_ok=True)
if os.path.isfile('PAKET_HASIL_ANGGOTA_A_V3.zip'):
    with zipfile.ZipFile('PAKET_HASIL_ANGGOTA_A_V3.zip') as z:
        z.extract('DATASET_V3_KANDIDAT_OPERASIONAL.csv',path='HASIL_V3')
elif os.path.isfile('DATASET_V3_KANDIDAT_OPERASIONAL.csv'):
    shutil.copyfile('DATASET_V3_KANDIDAT_OPERASIONAL.csv','HASIL_V3/DATASET_V3_KANDIDAT_OPERASIONAL.csv')
else:
    raise FileNotFoundError('Unggah ZIP paket V3 atau CSV kandidat V3 dahulu.')
print('Siap:',os.path.getsize('HASIL_V3/DATASET_V3_KANDIDAT_OPERASIONAL.csv'),'bytes')


In [ ]:
# Finalisasi statistik panjang IndoBERT dari CSV V3 (wajib untuk BAB 1 final).
# Gunakan TOKENIZER yang sama dengan Anggota B: indobenchmark/indobert-base-p1
import csv, json, collections, os
from pathlib import Path
from transformers import AutoTokenizer
from tqdm.auto import tqdm

DATA_PATH = 'HASIL_V3/DATASET_V3_KANDIDAT_OPERASIONAL.csv'
assert Path(DATA_PATH).exists(), 'File CSV V3 belum ditemukan, cek folder hasil ektraksi.'
tokenizer = AutoTokenizer.from_pretrained('indobenchmark/indobert-base-p1', use_fast=True)
total = collections.Counter(); over256 = collections.Counter(); lengths=collections.defaultdict(list)

with open(DATA_PATH, encoding='utf-8-sig', newline='') as f:
    reader = csv.DictReader(f)
    batch=[]
    for row in tqdm(reader, total=29189):
        batch.append((row['label'], row['text_clean']))
        if len(batch)==64:
            lengths_batch=tokenizer([s for _,s in batch], truncation=False, add_special_tokens=True, return_length=True)['length']
            for (label,_),ln in zip(batch,lengths_batch):
                total[label]+=1; over256[label]+=int(ln>256); lengths[label].append(ln)
            batch=[]
    if batch:
        lengths_batch=tokenizer([s for _,s in batch], truncation=False, add_special_tokens=True, return_length=True)['length']
        for (label,_),ln in zip(batch,lengths_batch):
            total[label]+=1; over256[label]+=int(ln>256); lengths[label].append(ln)

assert total['0']==20867 and total['1']==8322, f'Jumlah berbeda dari V3: {total}'
import statistics
res={
  'model_tokenizer': 'indobenchmark/indobert-base-p1',
  'token_count_includes_special_tokens': True,
  'threshold': 256,
  'class_0_VALID': {'n':total['0'],'count_over_256':over256['0'], 'percent_over_256':round(100*over256['0']/total['0'],2),'median_token':statistics.median(lengths['0'])},
  'class_1_HOAKS': {'n':total['1'],'count_over_256':over256['1'], 'percent_over_256':round(100*over256['1']/total['1'],2),'median_token':statistics.median(lengths['1'])}
}
Path('TOKEN_INDOBERT_V3.json').write_text(json.dumps(res,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(res,ensure_ascii=False,indent=2))


In [ ]:
from google.colab import files
files.download('TOKEN_INDOBERT_V3.json')
